# 🎓 Project 5: Bayes Decision Assistant
## SMS Spam Filter & Medical Test Risk Calculator
**MCA Semester I Mini Project: Mathematical Foundations with Python**  
**Team Roll Numbers:** 26 to 31  

---  
> **Note for Google Colab:** This notebook is completely self-contained. You can click **Runtime -> Run all** and it will automatically download the dataset, train the scratch Naive Bayes model, execute the Bayesian medical calculator, and run the 100,000-sample Monte Carlo simulation!

## 1. Setup & Automatic Dataset Download
We download the official UCI SMS Spam Collection dataset directly into the Colab environment.

In [ ]:
import urllib.request, zipfile, io, os, math, string, re
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Download UCI SMS Spam Dataset
url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(req) as resp:
    z = zipfile.ZipFile(io.BytesIO(resp.read()))
    with z.open("SMSSpamCollection") as f:
        lines = [line.decode("utf-8", errors="ignore").strip() for line in f if line.strip()]
        data = [l.split("	", 1) for l in lines if len(l.split("	", 1)) == 2]
        df = pd.DataFrame(data, columns=["label", "message"])

print(f"Dataset successfully loaded! Shape: {df.shape}")
df.head()

## 2. Text Preprocessing & Scratch Multinomial Naive Bayes
We clean text using regular expressions and implement **Multinomial Naive Bayes with Laplace Smoothing** from scratch without using scikit-learn.

In [ ]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    return re.sub(r"\s+", " ", text).strip()

def tokenize(text: str) -> list:
    cleaned = clean_text(text)
    return [w for w in cleaned.split() if len(w) > 1]

class NaiveBayesClassifier:
    def __init__(self, alpha: float = 1.0):
        self.alpha = alpha
        self.classes = ["ham", "spam"]
        self.class_counts = {c: 0 for c in self.classes}
        self.class_priors = {c: 0.0 for c in self.classes}
        self.word_counts = {c: defaultdict(int) for c in self.classes}
        self.total_words_in_class = {c: 0 for c in self.classes}
        self.vocab = set()
        self.vocab_size = 0

    def fit(self, messages, labels):
        total_samples = len(messages)
        for msg, label in zip(messages, labels):
            label = str(label).strip().lower()
            if label not in self.classes: continue
            self.class_counts[label] += 1
            tokens = tokenize(msg)
            for token in tokens:
                self.word_counts[label][token] += 1
                self.total_words_in_class[label] += 1
                self.vocab.add(token)
        self.vocab_size = len(self.vocab)
        for c in self.classes:
            self.class_priors[c] = self.class_counts[c] / total_samples
        return self

    def _log_likelihood(self, token, class_label):
        count = self.word_counts[class_label].get(token, 0)
        num = count + self.alpha
        den = self.total_words_in_class[class_label] + (self.alpha * self.vocab_size)
        return math.log(num / den)

    def predict_proba(self, message: str) -> dict:
        tokens = tokenize(message)
        log_scores = {}
        for c in self.classes:
            score = math.log(self.class_priors[c]) if self.class_priors[c] > 0 else -1e9
            for token in tokens:
                score += self._log_likelihood(token, c)
            log_scores[c] = score
        max_log = max(log_scores.values())
        exp_scores = {c: math.exp(score - max_log) for c, score in log_scores.items()}
        tot = sum(exp_scores.values())
        return {c: exp_scores[c] / tot for c in self.classes}

    def predict(self, message: str) -> str:
        p = self.predict_proba(message)
        return "spam" if p["spam"] >= p["ham"] else "ham"

## 3. Train-Test Split and Model Evaluation
We split into 80% train and 20% test using NumPy, evaluate metrics, and plot the confusion matrix.

In [ ]:
np.random.seed(42)
indices = np.random.permutation(len(df))
test_size = int(len(df) * 0.2)
train_df = df.iloc[indices[test_size:]].reset_index(drop=True)
test_df = df.iloc[indices[:test_size]].reset_index(drop=True)

model = NaiveBayesClassifier(alpha=1.0)
model.fit(train_df["message"].tolist(), train_df["label"].tolist())

y_test = test_df["label"].tolist()
y_pred = [model.predict(m) for m in test_df["message"].tolist()]

tp = sum(1 for yt, yp in zip(y_test, y_pred) if yt == "spam" and yp == "spam")
fp = sum(1 for yt, yp in zip(y_test, y_pred) if yt == "ham" and yp == "spam")
tn = sum(1 for yt, yp in zip(y_test, y_pred) if yt == "ham" and yp == "ham")
fn = sum(1 for yt, yp in zip(y_test, y_pred) if yt == "spam" and yp == "ham")

acc = (tp + tn) / len(y_test)
prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0

print(f"Accuracy : {acc*100:.2f}%")
print(f"Precision: {prec*100:.2f}%")
print(f"Recall   : {rec*100:.2f}%")
print(f"F1-Score : {f1:.4f}")

# Plot Confusion Matrix
plt.figure(figsize=(5, 4))
sns.heatmap([[tn, fp], [fn, tp]], annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred Ham", "Pred Spam"], yticklabels=["Actual Ham", "Actual Spam"])
plt.title("Confusion Matrix Heatmap (Naive Bayes)")
plt.show()

## 4. Medical Test Risk Calculator & Base Rate Neglect
We implement Bayes' Theorem to calculate (	ext{Disease} \mid +)$:

In [ ]:
def calculate_bayes_medical(sensitivity: float, specificity: float, prevalence: float):
    sens = sensitivity / 100.0
    spec = specificity / 100.0
    prev = prevalence / 100.0
    p_pos = (sens * prev) + ((1.0 - spec) * (1.0 - prev))
    ppv = (sens * prev) / p_pos
    return ppv * 100.0

sens = 95.0
spec = 90.0
prev = 1.0
ppv = calculate_bayes_medical(sens, spec, prev)
print(f"Sensitivity: {sens}%, Specificity: {spec}%, Prevalence: {prev}%")
print(f"P(Disease | Positive Test) = {ppv:.2f}%")
print(f"False Alarm Rate = {100.0 - ppv:.2f}%")

## 5. Monte Carlo Simulation (100,000 Individuals)
We use NumPy's random generator to simulate 100,000 individuals and confirm theoretical Bayes probability.

In [ ]:
np.random.seed(42)
N = 100000
has_disease = (np.random.rand(N) < (prev / 100.0)).astype(int)
rand_tests = np.random.rand(N)
test_pos = np.zeros(N, dtype=int)
test_pos[has_disease == 1] = (rand_tests[has_disease == 1] < (sens / 100.0)).astype(int)
test_pos[has_disease == 0] = (rand_tests[has_disease == 0] < (1.0 - (spec / 100.0))).astype(int)

sim_tp = int(np.sum((has_disease == 1) & (test_pos == 1)))
sim_fp = int(np.sum((has_disease == 0) & (test_pos == 1)))
sim_ppv = (sim_tp / (sim_tp + sim_fp)) * 100.0

print(f"Theoretical PPV : {ppv:.2f}%")
print(f"Monte Carlo PPV : {sim_ppv:.2f}%")
print(f"Difference      : {abs(sim_ppv - ppv):.4f}%")

# Comparison Bar Chart
plt.figure(figsize=(6, 4))
plt.bar(["Theoretical Bayes", "Monte Carlo (N=100k)"], [ppv, sim_ppv], color=["#2b5c8f", "#e26d5c"], width=0.4)
plt.ylabel("P(Disease | Positive) (%)")
plt.title("Empirical Verification of Bayes Theorem")
plt.ylim(0, max(ppv, sim_ppv) * 1.3)
plt.show()